# Issue #08 [Data/Filtering]: Selección Avanzada, Máscaras Booleanas e Invariantes Analíticos

**Subentorno:** `analytics/` | **Rama:** `feat/MPHR-analytics-nosql` | **Script equivalente:** `scripts/08_seleccionar_candidatos_prioritarios.py`

Este notebook segmenta a los jóvenes del dataset preparado en el Issue #05 para responder una pregunta concreta del área de selección: *¿a qué postulantes conviene llamar primero a entrevista?* Se trabaja con `.loc`, `.iloc`, `.isin()`, `.between()` y `.query()`, y cada resultado se blinda con aserciones `assert` antes de exportarlo (`CA-ISS-08-01` a `CA-ISS-08-04`).

In [1]:
from pathlib import Path
import json
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / 'data' / 'processed').exists():
    ANALYTICS_ROOT = cwd
elif (cwd / 'analytics' / 'data' / 'processed').exists():
    ANALYTICS_ROOT = cwd / 'analytics'
else:
    ANALYTICS_ROOT = cwd.parent

PREPARED_CSV = ANALYTICS_ROOT / 'data' / 'processed' / 'yeb_postulaciones_preparado.csv'
VACANTES_SEED = ANALYTICS_ROOT / 'data' / 'seeds' / 'vacantes.ejson'
REPORTE_CSV = ANALYTICS_ROOT / 'outputs' / 'candidatos_prioritarios_validados.csv'

df = pd.read_csv(PREPARED_CSV, parse_dates=['job_published_at', 'applied_at', 'updated_at'])
print('Raíz analítica resuelta :', ANALYTICS_ROOT.name)
print('Dataset preparado       :', df.shape)

Raíz analítica resuelta : analytics
Dataset preparado       : (93, 27)


## 1. Parámetros de negocio

Las habilidades priorizadas no se eligen a mano: se toman de `habilidades_requeridas` en las vacantes publicadas de la semilla documental del Issue #03. Así, si las empresas cambian lo que piden, el filtro cambia con ellas.

In [2]:
ESTADOS_ACTIVOS = ['submitted', 'under_review']
UMBRAL_PERFIL = 80
EDAD_MIN, EDAD_MAX, EDAD_MAYORIA = 16, 24, 18

vacantes = json.loads(VACANTES_SEED.read_text(encoding='utf-8'))
HABILIDADES = sorted({h for v in vacantes if v['estado'] == 'published' for h in v['habilidades_requeridas']})
print('Habilidades demandadas:', HABILIDADES)
print('Presentes en el dataset:', sorted(set(HABILIDADES) & set(df['primary_skill'])))

Habilidades demandadas: ['Atención al Cliente', 'Canva', 'Comunicación Asertiva', 'Control de Inventarios', 'Excel', 'Facturación Electrónica', 'Git', 'HTML/CSS', 'JavaScript', 'Python', 'SQL']
Presentes en el dataset: ['Atención al Cliente', 'Excel', 'Git', 'JavaScript', 'Python', 'SQL']


## 2. Cohortes de edad con `.loc` y máscaras booleanas (`CA-ISS-08-01`)

Los menores de 16 y 17 años tienen protección especial (Ley 1098 de 2006 y matriz ética del Issue #01), así que se separan de los jóvenes de 18 a 24 años. La partición debe ser exacta: si un registro no cae en ninguna cohorte, su edad y su bandera `is_protected_minor` se contradicen.

In [3]:
menores = df.loc[df['candidate_age'].between(EDAD_MIN, EDAD_MAYORIA - 1) & df['is_protected_minor']]
adultos = df.loc[df['candidate_age'].between(EDAD_MAYORIA, EDAD_MAX) & ~df['is_protected_minor']]

assert len(menores) + len(adultos) == len(df)
assert menores.index.intersection(adultos.index).empty

resumen = pd.DataFrame({
    'cohorte': ['Menor protegido (16-17)', 'Joven adulto (18-24)'],
    'postulaciones': [len(menores), len(adultos)],
    'con_permiso_verificado': [int(menores['minor_permit_verified'].sum()), int(adultos['minor_permit_verified'].sum())],
})
resumen

,cohorte,postulaciones,con_permiso_verificado
0,Menor protegido (16-17),21,21
1,Joven adulto (18-24),72,0


## 3. Filtros compuestos con `.isin()` y `.between()`

Cada condición se escribe entre paréntesis. En pandas, `&` y `|` tienen **más prioridad** que `==` o `>=`, así que olvidar los paréntesis cambia el significado de la consulta o produce un error (riesgo `TEC-013`). La siguiente celda lo demuestra a propósito:

In [4]:
try:
    df[df['status'] == 'submitted' & df['candidate_age'] >= 18]
except TypeError as exc:
    print('Sin paréntesis pandas evalúa primero  \'submitted\' & df[...]  ->', type(exc).__name__)

correcto = df[(df['status'] == 'submitted') & (df['candidate_age'] >= 18)]
print('Con paréntesis la consulta es válida  ->', len(correcto), 'registros')

Sin paréntesis pandas evalúa primero  'submitted' & df[...]  -> TypeError
Con paréntesis la consulta es válida  -> 16 registros


In [5]:
pasos = {
    'Dataset preparado': pd.Series(True, index=df.index),
    'Estado activo (.isin)': df['status'].isin(ESTADOS_ACTIVOS),
    'Habilidad demandada (.isin)': df['primary_skill'].isin(HABILIDADES),
    'Aspiración dentro del rango (.between)': df['expected_salary_cop'].between(df['salary_min_offered'], df['salary_max_offered']),
    f'Perfil >= {UMBRAL_PERFIL} %': df['profile_completeness_pct'] >= UMBRAL_PERFIL,
    'Menor solo con permiso (LEG-001)': (~df['is_protected_minor']) | (df['minor_permit_verified']),
}
acumulado = pd.Series(True, index=df.index)
embudo = []
for nombre, mascara in pasos.items():
    acumulado &= mascara
    embudo.append({'filtro': nombre, 'quedan': int(acumulado.sum())})
seleccion_mascaras = df.loc[acumulado]
pd.DataFrame(embudo)

,filtro,quedan
0,Dataset preparado,93
1,Estado activo (.isin),38
2,Habilidad demandada (.isin),29
3,Aspiración dentro del rango (.between),17
4,Perfil >= 80 %,10
5,Menor solo con permiso (LEG-001),10


## 4. La misma selección con `.query()` y parámetros `@variable` (`CA-ISS-08-02`)

`.query()` permite escribir la regla casi como se diría en voz alta. Los valores no se pegan dentro del texto sino que se pasan con `@`, de modo que cambiar un umbral no exige reescribir la consulta. Para comprobar que no hay errores de precedencia, el resultado debe coincidir registro por registro con la selección por máscaras.

In [6]:
estados, habilidades, umbral = ESTADOS_ACTIVOS, HABILIDADES, UMBRAL_PERFIL
edad_min, edad_max = EDAD_MIN, EDAD_MAX

seleccion_query = df.query(
    'status in @estados'
    ' and primary_skill in @habilidades'
    ' and salary_min_offered <= expected_salary_cop <= salary_max_offered'
    ' and profile_completeness_pct >= @umbral'
    ' and @edad_min <= candidate_age <= @edad_max'
    ' and (not is_protected_minor or minor_permit_verified)'
)
assert seleccion_query.index.equals(seleccion_mascaras.index)
print('Registros seleccionados con .query():', len(seleccion_query), '| idénticos a la selección por máscaras')

# .iloc para tomar las tres primeras posiciones del ranking por completitud
seleccion_query.sort_values(['profile_completeness_pct', 'application_id'], ascending=[False, True]).iloc[:3][
    ['application_id', 'candidate_age', 'primary_skill', 'profile_completeness_pct', 'status']]

Registros seleccionados con .query(): 10 | idénticos a la selección por máscaras


,application_id,candidate_age,primary_skill,profile_completeness_pct,status
9,APP-2026-0011,18,Excel,98,under_review
63,APP-2026-0081,19,SQL,97,under_review
22,APP-2026-0035,17,Excel,96,submitted


## 5. Invariantes con `assert` (`CA-ISS-08-03` / `CP-048B`)

Antes de exportar, el subconjunto final debe cumplir reglas que nunca pueden romperse. Si alguna falla, `AssertionError` detiene el proceso y el reporte no se genera. *(No ejecutar con `python -O`, porque esa opción elimina los `assert`.)*

In [7]:
def validar_invariantes(sub):
    assert sub['candidate_age'].between(EDAD_MIN, EDAD_MAX).all(), 'Edad fuera del rango permitido en la plataforma (16-24)'
    assert (sub['is_protected_minor'] == (sub['candidate_age'] < EDAD_MAYORIA)).all(), 'Bandera is_protected_minor contradice la edad declarada'
    assert sub.loc[sub['is_protected_minor'], 'minor_permit_verified'].all(), 'LEG-001: menor de edad sin permiso de trabajo verificado'
    assert not sub.loc[~sub['is_protected_minor'], 'minor_permit_verified'].any(), 'Permiso de menor marcado en un mayor de edad'
    assert (sub['salary_min_offered'] <= sub['salary_max_offered']).all(), 'Vacante con salario mínimo mayor que el máximo'
    assert ((sub['job_published_at'] <= sub['applied_at']) & (sub['applied_at'] <= sub['updated_at'])).all(), 'Cronología imposible'
    assert sub['application_id'].is_unique and sub['candidate_hash'].is_unique, 'Identificador repetido'
    assert not {'email_contacto', 'telefono_contacto'} & set(sub.columns), 'El reporte no puede llevar datos de contacto'
    return True

print('Subconjunto final válido:', validar_invariantes(seleccion_query))

Subconjunto final válido: True


Ahora se fuerza un registro inconsistente para comprobar que el pipeline realmente se detiene: un joven de 16 años marcado como menor pero **sin** permiso de trabajo verificado.

In [8]:
forzado = seleccion_query.iloc[[0]].copy()
forzado[['candidate_age', 'is_protected_minor', 'minor_permit_verified']] = [16, True, False]
forzado[['application_id', 'candidate_hash']] = ['APP-FORZADO-01', 'CAND-FORZADO-01']

try:
    validar_invariantes(pd.concat([seleccion_query, forzado]))
    print('El pipeline NO se detuvo: revisar invariantes')
except AssertionError as exc:
    print('Pipeline abortado correctamente ->', exc)

Pipeline abortado correctamente -> LEG-001: menor de edad sin permiso de trabajo verificado


## 6. Contraste con el reporte exportado (`CA-ISS-08-04` / `CP-048C`)

El script `08_seleccionar_candidatos_prioritarios.py` genera `outputs/candidatos_prioritarios_validados.csv` y la bitácora `outputs/bitacora_candidatos_prioritarios.md`. Esta última celda confirma que el notebook y el script llegan a los mismos postulantes.

In [9]:
reporte = pd.read_csv(REPORTE_CSV)
assert sorted(reporte['application_id']) == sorted(seleccion_query['application_id'])
assert not {'email_contacto', 'telefono_contacto'} & set(reporte.columns)
print('Reporte exportado:', reporte.shape, '| mismos application_id que el notebook')
reporte.groupby('cohorte').agg(priorizados=('application_id', 'count'), completitud_media=('profile_completeness_pct', 'mean')).round(1)

Reporte exportado: (10, 20) | mismos application_id que el notebook


,priorizados,completitud_media
cohorte,,
Joven adulto (18-24),6,92.2
Menor protegido (16-17),4,88.5
